In [10]:
import numpy as np
import pdfplumber
import pandas as pd
import re

In [11]:
def cargar_pdf(ruta_pdf: str):
    f = open(ruta_pdf, "rb")
    pdf = pdfplumber.open(f)
    return pdf, f

def extraer_texto_paginas(pdf) -> str:
    texto = ""
    for p in pdf.pages:
        contenido = p.extract_text()
        if contenido:
            texto += contenido + "\n"
    return texto

def limpiar_lineas(texto: str) -> list:
    lineas = [line.strip() for line in texto.split("\n") if line.strip()]
    return lineas

def extraer_transacciones(lineas: list) -> pd.DataFrame:
    patron = re.compile(
        r"(\d{2}/\d{2})\s+(.*?)\s+(\d{1,3}(?:\.\d{3})*,\d{2})"
    )

    registros = []
    for linea in lineas:
        match = patron.search(linea)
        if match:
            fecha, establecimiento, valor = match.groups()
            registros.append({
                "fecha": fecha,
                "establecimiento": establecimiento,
                "valor": float(valor.replace(".", "").replace(",", "."))
            })

    return pd.DataFrame(registros)

def cargar_estado_cuenta(ruta_pdf: str) -> pd.DataFrame:
    pdf, f = cargar_pdf(ruta_pdf)
    texto = extraer_texto_paginas(pdf)
    lineas = limpiar_lineas(texto)
    df = extraer_transacciones(lineas)
    pdf.close()
    f.close()
    return df

if __name__ == "__main__":
    ruta = "D:/python_projects/presupuesto/data/titanium.pdf"
    df = cargar_estado_cuenta(ruta)
    print(df)

    fecha                              establecimiento    valor
0   30/05  8119543 DIFERIDO INTERNACIONAL ONLINE (2/6)   671.70
1   30/05  8119546 DIFERIDO INTERNACIONAL ONLINE (2/6)   327.12
2   17/07  8169587 DIFERIDO INTERNACIONAL ONLINE (1/6)   122.29
3   28/11   994746 PTP - UNIVERSIDAD INTERNACIO (8/24)   110.80
4   14/05           7 DTF BLUECARD ECUADOR S A (FINAL)    72.66
5   16/07                            22125 VIA VENETTO    63.20
6   12/07                      8270782 DTV*DIRECTVGO 4    19.99
7   12/07            8270782 RET IVA SERV DIGITAL 100%     3.00
8   20/06                         5461510 UBER RIDES S     3.13
9   20/06            5461510 RET IVA SERV DIGITAL 100%     0.47
10  24/06                         6239776 UBER RIDES S     3.83
11  24/06            6239776 RET IVA SERV DIGITAL 100%     0.57
12  25/06                         6375874 UBER RIDES S     2.82
13  25/06            6375874 RET IVA SERV DIGITAL 100%     0.42
14  27/06                         637587

In [12]:
df.to_excel("output/estado_cuenta.xlsx", index=False)